In [ ]:
# Imports
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import storage

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import GridSearchCV



In [ ]:
# Load data from GCS
BUCKET_NAME = "myscrapers-qxa24002-v1"
DATA_KEY = "structured/datasets/listings_master_llm.csv"
TIMEZONE = "America/New_York"

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(DATA_KEY)

df = pd.read_csv(io.BytesIO(blob.download_as_bytes()))
df.head()

In [ ]:
#Basic Prep
def clean_numeric(s):
    s = s.astype(str).str.replace(r"[^\d.]+", "", regex=True).str.strip()
    return pd.to_numeric(s, errors="coerce")

df["scraped_at_dt_utc"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)
df["scraped_at_local"] = df["scraped_at_dt_utc"].dt.tz_convert(TIMEZONE)
df["date_local"] = df["scraped_at_local"].dt.date

df["price_num"] = clean_numeric(df["price"])
df["year_num"] = clean_numeric(df["year"])
df["mileage_num"] = clean_numeric(df["mileage"])

df = df[df["price_num"].notna()].copy()

df[["price_num", "year_num", "mileage_num"]].describe()


In [ ]:
#Time-based split: use past data to predict today’s listings
unique_dates = sorted(d for d in df["date_local"].dropna().unique())
today_local = unique_dates[-1]

train_df = df[df["date_local"] < today_local].copy()
test_df = df[df["date_local"] == today_local].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Today local:", today_local)

In [ ]:
#Metrics helper
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def bias(y_true, y_pred):
    return np.mean(y_pred - y_true)

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results = {
        "model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": rmse(y_test, preds),
        "MAPE": mape(y_test, preds),
        "Bias": bias(y_test, preds)
    }
    return model, preds, results

In [ ]:
#Baseline Model
baseline_cat = ["make", "model"]
baseline_num = ["year_num", "mileage_num"]
baseline_feats = baseline_cat + baseline_num

baseline_pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), baseline_num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), baseline_cat),
    ]
)

baseline_model = Pipeline([
    ("pre", baseline_pre),
    ("model", DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=42))
])

X_train_base = train_df[baseline_feats]
y_train = train_df["price_num"]
X_test_base = test_df[baseline_feats]
y_test = test_df["price_num"]

baseline_model, baseline_preds, baseline_results = evaluate_model(
    "Baseline Decision Tree",
    baseline_model,
    X_train_base, y_train,
    X_test_base, y_test
)

baseline_results

In [ ]:
#Improved model
improved_cat = [
    "make", "model", "transmission", "fuel_type",
    "condition", "title_status", "color"
]
improved_num = ["year_num", "mileage_num"]
improved_feats = improved_cat + improved_num

improved_pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), improved_num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), improved_cat),
    ]
)

improved_pipeline = Pipeline([
    ("pre", improved_pre),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [2, 5, 10]
}

grid = GridSearchCV(
    improved_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

X_train_imp = train_df[improved_feats]
X_test_imp = test_df[improved_feats]

grid.fit(X_train_imp, y_train)

best_improved_model = grid.best_estimator_
improved_preds = best_improved_model.predict(X_test_imp)

improved_results = {
    "model": "Improved Random Forest",
    "MAE": mean_absolute_error(y_test, improved_preds),
    "RMSE": rmse(y_test, improved_preds),
    "MAPE": mape(y_test, improved_preds),
    "Bias": bias(y_test, improved_preds)
}

print("Best params:", grid.best_params_)
improved_results

In [ ]:
#Compare baseline vs improved
comparison = pd.DataFrame([baseline_results, improved_results])
comparison

In [ ]:
#Save predictions
preds_out = test_df[[
    "post_id", "scraped_at", "make", "model", "year", "mileage",
    "transmission", "fuel_type", "condition", "title_status", "color"
]].copy()

preds_out["actual_price"] = y_test.values
preds_out["baseline_pred_price"] = np.round(baseline_preds, 2)
preds_out["improved_pred_price"] = np.round(improved_preds, 2)

preds_out.head()

In [ ]:
# Permutation importance for improved model
perm = permutation_importance(
    best_improved_model,
    X_test_imp,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "feature": improved_feats,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance_df

In [ ]:
top_imp = importance_df.sort_values("importance_mean", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(top_imp["feature"], top_imp["importance_mean"])
plt.title("Permutation Importance - Improved Model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
#PDP plots for top 3 features
top3_features = importance_df.head(3)["feature"].tolist()
top3_features

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
PartialDependenceDisplay.from_estimator(
    best_improved_model,
    X_test_imp,
    features=top3_features,
    ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
#Track performance as dataset grows
growth_rows = []
df_sorted = df.sort_values("scraped_at_dt_utc").copy()

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

for frac in fractions:
    subset = df_sorted.iloc[:int(len(df_sorted) * frac)].copy()
    unique_dates_subset = sorted(d for d in subset["date_local"].dropna().unique())

    if len(unique_dates_subset) < 2:
        continue

    subset_today = unique_dates_subset[-1]
    subset_train = subset[subset["date_local"] < subset_today].copy()
    subset_test = subset[subset["date_local"] == subset_today].copy()

    if len(subset_train) < 40 or len(subset_test) == 0:
        continue

    X_tr = subset_train[improved_feats]
    y_tr = subset_train["price_num"]
    X_te = subset_test[improved_feats]
    y_te = subset_test["price_num"]

    best_improved_model.fit(X_tr, y_tr)
    preds = best_improved_model.predict(X_te)

    growth_rows.append({
        "data_fraction": frac,
        "train_rows": len(subset_train),
        "test_rows": len(subset_test),
        "MAE": mean_absolute_error(y_te, preds),
        "RMSE": rmse(y_te, preds),
        "MAPE": mape(y_te, preds),
        "Bias": bias(y_te, preds)
    })

growth_df = pd.DataFrame(growth_rows)
growth_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(growth_df["data_fraction"], growth_df["MAE"], marker="o", label="MAE")
plt.plot(growth_df["data_fraction"], growth_df["RMSE"], marker="o", label="RMSE")
plt.xlabel("Fraction of Data Used")
plt.ylabel("Error")
plt.title("Model Performance as Dataset Grows")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optional: save notebook artifacts
comparison.to_csv("comparison_metrics.csv", index=False)
importance_df.to_csv("permutation_importance.csv", index=False)
preds_out.to_csv("predictions_comparison.csv", index=False)
growth_df.to_csv("growth_metrics.csv", index=False)